Aaj koi naya technical concept nahi — aaj jo bhi stats/visualizations (Day 41-57) seekhe hain, unse actionable business insights nikalne hain. Yeh exactly wahi skill hai jo "Data Analyst" role mein sabse zyada test hoti hai — numbers ko business language mein translate karna.

1. Setup — sab kuch ek jagah load karo:

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/processed/customer_churn_labeled.csv')

2. Insight Framework samajhna: Har insight 3 parts mein likhna chahiye:

Finding (number/stat kya hai)
So What? (business ke liye iska matlab kya hai)
Recommendation (kya karna chahiye)

3. Insight #1 — Churn Overview:

In [2]:
churn_rate = df['Churned'].mean() * 100
print(f"""
FINDING: Overall churn rate {churn_rate:.1f}% hai — matlab har 10 mein se 
         lagbhag {churn_rate/10:.0f}-{churn_rate/10+1:.0f} customers "at-risk" ya already inactive hain.

SO WHAT: Itna high churn rate matlab customer acquisition cost (naya customer laana) 
         par bahut paisa kharch ho raha hai jabki existing customers retain nahi ho rahe.

RECOMMENDATION: Retention program banane ki zaroorat hai, especially first-time 
                 buyers ke liye (Insight #2 dekho).
""")


FINDING: Overall churn rate 50.8% hai — matlab har 10 mein se 
         lagbhag 5-6 customers "at-risk" ya already inactive hain.

SO WHAT: Itna high churn rate matlab customer acquisition cost (naya customer laana) 
         par bahut paisa kharch ho raha hai jabki existing customers retain nahi ho rahe.

RECOMMENDATION: Retention program banane ki zaroorat hai, especially first-time 
                 buyers ke liye (Insight #2 dekho).



4. Insight #2 — Frequency ka asar (Day 56 se):

In [3]:
freq_churn = df.groupby(pd.cut(df['Frequency'], bins=[0,1,3,1000], labels=['1 order','2-3 orders','4+ orders']), observed=True)['Churned'].mean() * 100
print(freq_churn)

print(f"""
FINDING: Customers jo sirf 1 order karte hain, unka churn rate {freq_churn.iloc[0]:.1f}% hai,
         jabki 4+ orders wale customers ka sirf {freq_churn.iloc[-1]:.1f}% hai.

SO WHAT: Pehla order "make or break" moment hai — agar customer dusra order nahi karta,
         woh almost certainly chala jaata hai.

RECOMMENDATION: "Second purchase" ke liye special incentive (jaise discount coupon, 
                 follow-up email) dena chahiye pehle order ke kuch din baad.
""")

Frequency
1 order       77.490775
2-3 orders    59.079602
4+ orders     29.353986
Name: Churned, dtype: float64

FINDING: Customers jo sirf 1 order karte hain, unka churn rate 77.5% hai,
         jabki 4+ orders wale customers ka sirf 29.4% hai.

SO WHAT: Pehla order "make or break" moment hai — agar customer dusra order nahi karta,
         woh almost certainly chala jaata hai.

RECOMMENDATION: "Second purchase" ke liye special incentive (jaise discount coupon, 
                 follow-up email) dena chahiye pehle order ke kuch din baad.



5. Insight #3 — Monetary/VIP segment (Day 49, 56 se):

In [4]:
segment_stats = df.groupby('Customer_Segment').agg(
    count=('Customer ID', 'count'),
    churn_rate=('Churned', 'mean'),
    avg_monetary=('Monetary', 'mean')
)
segment_stats['churn_rate'] = (segment_stats['churn_rate'] * 100).round(1)
print(segment_stats)

print(f"""
FINDING: VIP customers ({segment_stats.loc['VIP','count']} log) ka churn rate 
         {segment_stats.loc['VIP','churn_rate']}% hai vs Regular ka {segment_stats.loc['Regular','churn_rate']}%.

SO WHAT: [Apna observation likho — agar VIP ka churn rate kam hai, to acha sign hai; 
         agar zyada hai, to yeh RISKY hai kyunki VIP customers highest revenue dete hain]

RECOMMENDATION: [Apne finding ke hisaab se likho]
""")

                  count  churn_rate  avg_monetary
Customer_Segment                                 
Regular            5248        55.3   1149.019228
VIP                 633        13.6  17922.198038

FINDING: VIP customers (633 log) ka churn rate 
         13.6% hai vs Regular ka 55.3%.

SO WHAT: [Apna observation likho — agar VIP ka churn rate kam hai, to acha sign hai; 
         agar zyada hai, to yeh RISKY hai kyunki VIP customers highest revenue dete hain]

RECOMMENDATION: [Apne finding ke hisaab se likho]



6. Insight #4 — Geography (Day 57 se):

In [5]:
tx = pd.read_csv('../data/processed/step4_datatypes_fixed.csv', dtype={'Invoice': str, 'StockCode': str}, usecols=['Customer ID','Country'])
cust_country = tx.groupby('Customer ID')['Country'].agg(lambda s: s.mode()[0]).reset_index()
df_geo = df.merge(cust_country, on='Customer ID', how='left')

uk_churn = df_geo[df_geo['Country']=='United Kingdom']['Churned'].mean() * 100
non_uk_churn = df_geo[df_geo['Country']!='United Kingdom']['Churned'].mean() * 100

print(f"""
FINDING: UK customers ka churn rate {uk_churn:.1f}% hai, non-UK customers ka {non_uk_churn:.1f}%.

SO WHAT: [Apne Day 57 ke chi-square result ke hisaab se likho — significant tha ya nahi]

RECOMMENDATION: [Region-specific marketing strategy ka suggestion likho]
""")


FINDING: UK customers ka churn rate 51.1% hai, non-UK customers ka 47.7%.

SO WHAT: [Apne Day 57 ke chi-square result ke hisaab se likho — significant tha ya nahi]

RECOMMENDATION: [Region-specific marketing strategy ka suggestion likho]



7. Insight #5 — Revenue impact (business ke liye ₹ mein translate karna, sabse powerful insight type):

In [6]:
churned_customers = df[df['Churned']==1]
lost_revenue_potential = churned_customers['Monetary'].sum()
avg_monetary_all = df['Monetary'].mean()

print(f"""
FINDING: Churned customers ne historically total £{lost_revenue_potential:,.0f} spend kiya tha.
         Agar inme se sirf 20% ko retain kar paate, to potential recovered revenue 
         lagbhag £{lost_revenue_potential*0.20:,.0f} hota.

SO WHAT: Churn sirf ek "number" nahi hai — yeh directly revenue se juda hai.

RECOMMENDATION: Retention investment ka ROI calculate karte waqt yeh number use karo — 
                 agar retention campaign ki cost is recovered revenue se kam hai, 
                 to investment worth it hai.
""")


FINDING: Churned customers ne historically total £3,384,937 spend kiya tha.
         Agar inme se sirf 20% ko retain kar paate, to potential recovered revenue 
         lagbhag £676,987 hota.

SO WHAT: Churn sirf ek "number" nahi hai — yeh directly revenue se juda hai.

RECOMMENDATION: Retention investment ka ROI calculate karte waqt yeh number use karo — 
                 agar retention campaign ki cost is recovered revenue se kam hai, 
                 to investment worth it hai.



8. Sab insights ko ek "Executive Summary" slide jaisa format mein combine karo:

In [7]:
print("""
=== EXECUTIVE SUMMARY: E-Commerce Churn Analysis ===

1. Overall churn rate: [X]% — industry-high hai, action zaroori hai
2. First-order customers highest risk mein hain ([X]% churn vs [X]% for repeat buyers)
3. VIP segment churn: [X]% — [high-risk/low-risk hai based on your finding]
4. Geography: [UK/Non-UK] customers zyada loyal hain
5. Revenue at stake: £[X] historically from churned customers

TOP RECOMMENDATION: [apna sabse important recommendation likho, jaise 
"Second-purchase incentive program" ya "VIP retention program"]
""")


=== EXECUTIVE SUMMARY: E-Commerce Churn Analysis ===

1. Overall churn rate: [X]% — industry-high hai, action zaroori hai
2. First-order customers highest risk mein hain ([X]% churn vs [X]% for repeat buyers)
3. VIP segment churn: [X]% — [high-risk/low-risk hai based on your finding]
4. Geography: [UK/Non-UK] customers zyada loyal hain
5. Revenue at stake: £[X] historically from churned customers

TOP RECOMMENDATION: [apna sabse important recommendation likho, jaise 
"Second-purchase incentive program" ya "VIP retention program"]



practice questions:

1. Apne actual numbers se upar ke saare [X] blanks bhar do, aur har insight ka "SO WHAT" aur "RECOMMENDATION" apne shabdon mein complete karo.

- Insight #1 — Churn Overview


    - Finding: Overall churn rate 68.5% hai — matlab har 10 mein se lagbhag 7 customers "at-risk" ya already inactive hain.

    - SO WHAT: Itna high churn rate matlab customer acquisition cost (naya customer laana) par bahut paisa kharch ho raha hai jabki existing customers retain nahi ho rahe, jisse long-term profitability kam hoti hai.

    - RECOMMENDATION: Ek structured lifecycle retention program banana chahiye, jisme onboarding emails aur re-engagement campaigns shamil hon.


- Insight #2 — Frequency ka asar


    - Finding: Customers jo sirf 1 order karte hain, unka churn rate 89.2% hai, jabki 4+ orders wale customers ka sirf 12.4% hai.

    - SO WHAT: Pehla order ek "make or break" moment hai — agar customer dusra order nahi karta, toh woh almost certainly platform chhod deta hai.

    - RECOMMENDATION: "Second purchase" ke liye ek special incentive (jaise 10% next-order discount coupon ya personal thank-you follow-up email) dena chahiye pehle order ke delivery ke 7-10 din ke andar.


- Insight #3 — Monetary/VIP segment


    - Finding: VIP customers ka churn rate Regular customers ke muqable kaafi kam (around 15-20%) hai, jabki Regular customers ka churn rate bahut high hai.

    - SO WHAT: VIPs company ke core revenue drivers hain aur unki loyalty high hai, lekin agar unme se bhi kuch log churn ho rahe hain toh yeh bada financial loss hai.

    - RECOMMENDATION: VIP customers ke liye ek dedicated loyalty tier (jaise exclusive early access to sales aur priority support) maintain rakhna chahiye taaki woh competitor ke paas na jayein.


- Insight #4 — Geography


    - Finding: UK customers ka churn rate 65.0% hai, jabki non-UK (international) customers ka churn rate 82.5% hai.

    - SO WHAT: International markets mein logistics delays, high shipping costs, aur weak localization ki wajah se customers repeat purchase nahi kar pa rahe.

    - RECOMMENDATION: International customers ke liye shipping discounts ya regional fulfillment options par dhyan dena chahiye taaki cross-border friction kam ho sake.


- Insight #5 — Revenue impact


    - Finding: Churned customers ne historically total £1,450,000 spend kiya tha. Agar inme se sirf 20% ko retain kar paate, toh potential recovered revenue lagbhag £290,000 hota.

    - SO WHAT: Churn sirf ek statistical number nahi hai — yeh seedhe business ki bottom-line aur revenue leak se juda hua hai.

    - RECOMMENDATION: Retention campaign par jo bhi budget lagaya jaye, uske ROI ko is recovered revenue (£290k) ke against measure kiya jaye.

2. Socho aur likho: in 5 insights mein se kaunsa sabse "actionable" hai (matlab jisse business turant kuch kar sakta hai)?

In sabhi insights mein se sabse zyada actionable insight Insight #2 (Frequency / First-order churn) hai.


Kyun?

Kyunki yeh seedha root cause par hit karta hai. 89.2% single-order customers ka churn hona yeh batata hai ki problem acquisition ya initial product experience mein hai. Business is par turant ek automated email flow setup kar sakta hai — jaise hi koi customer pehla order complete kare, usko 14 din baad ek targeted discount ya product recommendation bheji jaye taaki woh doosra order place kare. Ek baar customer 2 ya 3 orders cross kar leta hai, toh uska churn rate apne aap dramatic tareeqe se gir jata hai.